In [1]:
from pathlib import Path
import torch, torch.nn as nn
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import PartialState
from trl import PPOConfig, PPOTrainer

class DummyBackbone(nn.Module):
    def __init__(self, h: int = 16):
        super().__init__()
        self.emb = nn.Embedding(1, h)       
    def forward(self, input_ids=None, **kw):
        hs = self.emb(torch.zeros_like(input_ids))
        return type("DummyOut", (), {"hidden_states": [hs]})

class DummyValueOrReward(nn.Module):
    base_model_prefix = "backbone"
    def __init__(self, h: int = 16):
        super().__init__()
        self.backbone = DummyBackbone(h)
        self.score    = nn.Linear(h, 1)         
    def forward(self, *args, **kw):
        return self.backbone(*args, **kw)

cfg = PPOConfig(
    output_dir="ppo_deepseek_dummy",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    total_episodes=1,
    response_length=32,
    learning_rate=1e-5,
    vf_coef=0.0,
    kl_coef=0.02,
    seed=42,
    use_cpu=True,
    bf16=False,
    fp16=False,
    num_ppo_epochs = 3,
    report_to="wandb",  
    logging_steps=1,       
    run_name="deepseek-ppo",
)


device   = "mps" if torch.backends.mps.is_available() else "cpu"
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

policy    = AutoModelForCausalLM.from_pretrained(model_id,
                                                 torch_dtype="auto").to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
tokenizer.add_special_tokens({"pad_token": tokenizer.eos_token})


reward_model = DummyValueOrReward().to(device)
value_model  = DummyValueOrReward().to(device)


raw_ds = load_dataset("squad", split="train[:1%]")
def to_features(example):
    prompt = f"### Question: {example['question']}\n### Answer:"
    ids    = tokenizer(prompt, return_tensors="pt").input_ids[0]
    return {"input_ids": ids, "lengths": len(ids)}
with PartialState().local_main_process_first():
    ds = (raw_ds
          .map(to_features,
               remove_columns=raw_ds.column_names,
               num_proc=4)
          .filter(lambda e: e["lengths"] < 256, num_proc=4))


trainer = PPOTrainer(
    args             = cfg,
    processing_class = tokenizer,
    model            = policy,
    ref_model        = None,  
    reward_model     = reward_model,
    value_model      = value_model,
    train_dataset    = ds,
    eval_dataset     = None
)
trainer.train()
trainer.save_model(cfg.output_dir)
print("Finished PPO with dummy reward/value models")


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


===training policy===


wandb: Currently logged in as: eliray (eliray-queen-mary-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


TypeError: object of type 'NoneType' has no len()

In [2]:
import os
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import PPOConfig, PPOTrainer
import numpy as np


class DummyRewardModel(nn.Module):
    """Simple reward model that returns random rewards for demonstration."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.linear = nn.Linear(config.hidden_size, 1)
        
    def forward(self, input_ids, attention_mask=None):
        # Return random rewards for now
        batch_size = input_ids.shape[0]
        rewards = torch.randn(batch_size, 1)
        return rewards

class DummyValueModel(nn.Module):
    """Simple value model for PPO (will be disabled with vf_coef=0)."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.base_model_prefix = "transformer"  # Required by PPO trainer
        self.linear = nn.Linear(config.hidden_size, 1)
        
    def score(self, hidden_states):
        # Return dummy values
        return self.linear(hidden_states)
        
    def forward(self, input_ids, attention_mask=None):
        # Return dummy hidden states
        batch_size, seq_len = input_ids.shape
        hidden_states = torch.randn(batch_size, seq_len, self.config.hidden_size)
        return type('DummyOutput', (), {'hidden_states': [hidden_states]})()


def get_reward(model, input_ids, pad_token_id, context_length):
    """Get rewards from the reward model."""
    with torch.no_grad():
        rewards = model(input_ids)
    # Return rewards, scores, and sequence_lengths (dummy values)
    return rewards, rewards.squeeze(-1), torch.ones_like(rewards.squeeze(-1))


model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"


print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


print("Loading model on CPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="cpu",
    low_cpu_mem_usage=True
)


print("Creating reference model...")
ref_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="cpu",
    low_cpu_mem_usage=True
)

print("Creating reward and value models...")
config = model.config
reward_model = DummyRewardModel(config)
value_model = DummyValueModel(config)

print("Creating training dataset...")
prompts = [
    "What is machine learning?",
    "Explain quantum computing.",
    "How does photosynthesis work?",
    "What are the benefits of exercise?",
    "Describe the water cycle.",
    "What is artificial intelligence?",
    "How do vaccines work?",
    "Explain the theory of relativity.",
    "What causes climate change?",
    "How does the internet work?",
] * 10

tokenized_prompts = tokenizer(
    prompts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

train_dataset = Dataset.from_dict({
    "input_ids": tokenized_prompts["input_ids"],
    "attention_mask": tokenized_prompts["attention_mask"]
})

eval_dataset = Dataset.from_dict({
    "input_ids": tokenized_prompts["input_ids"][:5],
    "attention_mask": tokenized_prompts["attention_mask"][:5]
})


ppo_config = PPOConfig(

    exp_name="ppo_deepseek_cpu",
    reward_model_path="dummy",
    num_ppo_epochs=4,
    whiten_rewards=False,
    kl_coef=0.05,
    cliprange=0.2,
    vf_coef=0.0,
    cliprange_value=0.2,
    gamma=1.0,
    lam=0.95,
    ds3_gather_for_generation=False,
    
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_mini_batches=1,
    
    response_length=64,
    temperature=0.7,
    
    fp16=False, 
    bf16=False,
    
    # Logging
    report_to=["wandb"], 
    logging_steps=1, 
    logging_first_step=True,
    output_dir="./ppo_output",
    

    seed=42,
    remove_unused_columns=False,
)

print("Initializing PPO trainer...")
trainer = PPOTrainer(
    args=ppo_config,
    processing_class=tokenizer,
    model=model,
    ref_model=ref_model,
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

from transformers import GenerationConfig
model.generation_config = GenerationConfig(
    max_new_tokens=64,
    temperature=0.7,
    top_k=0,
    top_p=1.0,
    do_sample=True,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

import sys
sys.modules[trainer.__module__].get_reward = get_reward

print("Starting PPO training...")
print(f"Total episodes: {ppo_config.total_episodes}")
print(f"Batch size: {ppo_config.batch_size}")
print(f"Logging to: {ppo_config.report_to}")

trainer.train()

print("Saving final model...")
trainer.save_model(ppo_config.output_dir)
tokenizer.save_pretrained(ppo_config.output_dir)

print("Training completed!")

Loading tokenizer...
Loading model on CPU...
Creating reference model...
Creating reward and value models...
Creating training dataset...
Initializing PPO trainer...


AttributeError: 'DummyValueModel' object has no attribute 'transformer'

In [ ]:
import torch
import torch.nn as nn
from types import SimpleNamespace
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorWithPadding,
)
from trl import PPOConfig, PPOTrainer

class DummyHead(nn.Module):
    def __init__(self, backbone, hidden_size):
        super().__init__()
        prefix = backbone.base_model_prefix
        self.base_model_prefix = prefix
        setattr(self, prefix, backbone)
        self.hidden_size = hidden_size

    def score(self, hidden_states: torch.Tensor) -> torch.Tensor:
        b, seq_len, _ = hidden_states.shape
        return torch.zeros(b, seq_len, device=hidden_states.device)

def make_dummy_module(policy_model):
    prefix = policy_model.base_model_prefix
    backbone = getattr(policy_model, prefix)
    hidden_size = policy_model.config.hidden_size
    return DummyHead(backbone, hidden_size)

device = torch.device("cpu")

model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="right")
if tokenizer.pad_token_id is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
policy_model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

reward_model = make_dummy_module(policy_model).to(device)
value_model  = make_dummy_module(policy_model).to(device)

texts = [
    "Hello, world!",
    "What is the capital of France?",
    "Tell me a cat joke."
] * 20
ds = Dataset.from_dict({"text": texts})
def tokenize_fn(ex):
    tok = tokenizer(ex["text"], truncation=True, max_length=64)
    return {"input_ids": tok["input_ids"], "attention_mask": tok["attention_mask"]}
ds = ds.map(tokenize_fn, batched=True).remove_columns("text")
ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

collator = DataCollatorWithPadding(tokenizer)

ppo_config = PPOConfig(
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    num_ppo_epochs=3,
    total_episodes=50,
    mini_batch_size=2,
    vf_coef=0.0,
    use_cpu=True,
)

trainer = PPOTrainer(
    args=ppo_config,
    processing_class=tokenizer,
    model=policy_model,
    ref_model=None,     
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=ds,
    data_collator=collator,
)

trainer.train()


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

===training policy===
